In [2]:
# !pip install langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [3]:
from langchain_core.documents import Document

In [4]:
sample_doc=Document(
    page_content="hello world",
    metadata={"source":"https://www.google.com"}
)

In [5]:
sample_doc

Document(metadata={'source': 'https://www.google.com'}, page_content='hello world')

In [6]:
type(sample_doc)

langchain_core.documents.base.Document

In [7]:
#text data
from langchain_community.document_loaders.text import TextLoader

loader=TextLoader("data/Python.txt",encoding="utf-8")

C:\Users\Zohra Fathima\AppData\Local\Temp\ipykernel_10916\1841854871.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.text import TextLoader


In [8]:
document=loader.load()

In [9]:
# from langchain_community.document_loaders.pdf import PyPDFLoader

# pdf_loader=PyPDFLoader("data/research.pdf")

# document=pdf_loader.load()

In [10]:
# from langchain_community.document_loaders.pdf import PyMuPDFLoader

# pdf_loader=PyMuPDFLoader("data/research.pdf")

# document=pdf_loader.load()

## Ingestion pipeline


### documents

In [11]:
# data=> documents
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

In [12]:
def laod_all_pdfs():
    folder_path="data/pdfs"
    num_docs=0
    all_docs=[]

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            # complete file path
            pdf_path=os.path.join(folder_path,filename)

            loader=PyPDFLoader(pdf_path)
            doc=loader.load()
            
            all_docs.extend(doc)
            num_docs+=1

    print("total pdfs:",num_docs)
    print("total pages:",len(all_docs))
    return all_docs

In [13]:
all_pdf_documents=laod_all_pdfs()

total pdfs: 2
total pages: 29


In [14]:
type(all_pdf_documents[1])

langchain_core.documents.base.Document

### chunks

In [15]:
# !pip install langchain_text_splitters

In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents,chunk_size=500,chunk_overlap=50):
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    chunked_docs=text_splitter.split_documents(documents)
    return chunked_docs

In [17]:
chunks=split_docs(all_pdf_documents)

In [18]:
len(chunks)

316

### embedding

In [19]:
from sentence_transformers import SentenceTransformer

In [20]:
class EmbeddingManager:
    def __init__(self,model_name="all-MiniLM-L6-v2"):
        
        self.model_name=model_name
        print("loading model....",self.model_name)
        self.model=SentenceTransformer(self.model_name)
        print("embedding dimensions=",self.model.get_sentence_embedding_dimension())


    def generate_embeddings(self,text):
        embeddings=self.model.encode(text,show_progress_bar=True)
        print("embeddings shape:",embeddings.shape)
        return embeddings
    
        
    

In [21]:
embedding_manager=EmbeddingManager()

loading model.... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embedding dimensions= 384


C:\Users\Zohra Fathima\AppData\Local\Temp\ipykernel_10916\3657386670.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dimensions=",self.model.get_sentence_embedding_dimension())


### vector store

In [22]:
import chromadb
import uuid

In [23]:
class VectorStoreManager:
    def __init__(self,persist_directory="data/vector_store",collection_name="pdf_documents"):
        self.collection_name=collection_name
        self.persist_directory=persist_directory
        self.collection=None
        self.client=None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory,exist_ok=True)

        #create client
        self.client=chromadb.PersistentClient(path=self.persist_directory)

        #create collection
        self.collection=self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description":"vector store collection for pdf embeddings in RAG"}
        )

        print("initailzed the vector sore with collection:",self.collection_name)
        print("docs in collection:", self.collection.count())


    def add_documents(self,documents,embeddings):
        if len(documents) != len(embeddings):
            raise ValueErroe("number of documents does not match to num of embeddings")


        #store=> ids,embedding,document,metadata
        ids=[]
        all_metadata=[]
        documents_content=[]
        embeddings_list=[]

        for i , (doc,embedding) in enumerate(zip(documents,embeddings)):
            doc_id=f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata=dict(doc.metadata)
            metadata["doc_index"]=i
            metadata["content_length"]=len(doc.page_content)
            all_metadata.append(metadata)


            documents_content.append(doc.page_content)

            embeddings_list.append(embedding.tolist())


            self.collection.add(
                ids=ids,
                metadatas=all_metadata,
                documents=documents_content,
                embeddings=embeddings_list
            )

        print("total doucmnets added in vector store :",len(documents_content))
        print("docs in collection: ",self.collection.count())

In [24]:
vector_store=VectorStoreManager()

initailzed the vector sore with collection: pdf_documents
docs in collection: 632


In [25]:
# data=>documents =>chunks=>embeddings=>store in vector stores

texts=[doc.page_content for doc in chunks]

embedding=embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks,embedding)

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

embeddings shape: (316, 384)
total doucmnets added in vector store : 316
docs in collection:  948


## retrieval pipeline

In [26]:
from sklearn.metrics.pairwise import cosine_similarity

In [27]:
class RAGRetriever:
    def __init__(self,embedding_manager,vector_store):
        self.embedding_manager=embedding_manager
        self.vector_store=vector_store

    def retrieve(self,query,top_k=5,score_threshold=0.0):

        #query=>embedding
        query_embeddings=self.embedding_manager.generate_embeddings([query])[0]

        #semantic search
        results=self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()],
            n_results=top_k
        )


        #cosine similarity
        retrieved_docs=[]
        if results["documents"] and results["documents"][0]:
            ids=results["ids"][0]
            metadatas=results["metadatas"][0]
            documents=results["documents"][0]
            distances=results["distances"][0]

            for i,(doc_id,metadata,document,distance) in enumerate(zip(ids,metadatas,documents,distances)):
                similarity_score=1-distance
                if similarity_score>=score_threshold:
                    retrieved_docs.append({
                        "id":doc_id,
                        "document":document,
                        "metadata":metadata,
                        "distance":distance,
                        "similarity_score":similarity_score,
                        "rank":i+1
                    })

            print(f"retrieved {len(retrieved_docs)} documents")

        else:
            print("no documents found")

        return retrieved_docs

In [28]:
rag_retriever=RAGRetriever(embedding_manager,vector_store)

In [29]:
rag_retriever.retrieve("what is llms")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 5 documents


[{'id': 'doc_364e5c1b-82cf-4e27-ae68-746fe9d68ce7',
  'document': 'recent researches towards utilizing content generated by LLMs\nthemselves for retrieval and enhancement purposes.',
  'metadata': {'creationdate': '2024-03-28T00:54:45+00:00',
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
   'creator': 'LaTeX with hyperref',
   'content_length': 112,
   'doc_index': 121,
   'author': '',
   'title': '',
   'moddate': '2024-03-28T00:54:45+00:00',
   'page_label': '5',
   'subject': '',
   'trapped': '/False',
   'page': 4,
   'source': 'data/pdfs\\research2.pdf',
   'total_pages': 21,
   'keywords': '',
   'producer': 'pdfTeX-1.40.25'},
  'distance': 0.7739307284355164,
  'similarity_score': 0.22606927156448364,
  'rank': 1},
 {'id': 'doc_023ceb57-a4d9-4035-973f-0deb1c8087e3',
  'document': 'recent researches towards utilizing content generated by LLMs\nthemselves for retrieval and enhancement purposes.',
  'metadata': {'a

# Integrate with LLMs

## Google-GEMINI

In [67]:
%pip install -U langchain-google-genai python-dotenv

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 1.5/1.5 MB 8.2 MB/s  0:00:00

  Attempting uninstall: python-dotenv

    Found existing installation: python-dotenv 1.2.1

    Uninstalling python-dotenv-1.2.1:

      Successfully uninstalled python-dotenv-1.2.1

   ----------- ---------------------------- 2/7 [pyasn1]
   ----------------- ---------------------- 3/7 [pyasn1-modules]
   ----------------- ---------------------- 3/7 [pyasn1-modules]
   ---------------------- ----------------- 4/7 [google-auth]
   ---------------------------- ----------- 5/7 [google-genai]
   ---------------------------- ----------- 5/7 [google-genai]
   ---------------------------- ----------- 5/7 [google-genai]
   ---------------------------- ----------- 5/7 [google-genai]
   ---------------------------- ----------- 5/7 [google-genai]
   ---------------------------- ----------- 5/7 [google-genai]
   ---------------------------- ----------- 5/

In [68]:
from dotenv import load_dotenv
import os
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

llm=ChatGoogleGenerativeAI(
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    model="gemini-3.8-flash",
    temperature=0.1,
    max_tokens=1024
)


In [81]:
# generate out retrival-augmented output
def generate_output(query,rag_retriever,llm,top_k=3):
    results=rag_retriever.retrieve(query,top_k)

    context="\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant content for your input")

    #context+query
    prompt= f""" use given context to generate the answer for the query
                Context:{context}
                Query:{query}"""

    response=llm.invoke(prompt) #expecting a string as prompt
    
    return response.text

In [82]:
answer=generate_output("What are the different paradigms of RAG discussed in the survey??",rag_retriever,llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 3 documents


In [83]:
print(answer)

Based on the provided context, the only paradigm explicitly mentioned is **naive RAG** (the text cuts off after "including naive RAG," and does not list the others).


## GROQ

In [86]:
# !pip install langchain-groq

In [92]:
from langchain_groq import ChatGroq

llm=ChatGroq(
    groq_api_key=os.getenv("GROQ_API_KEY"),
    model="openai/gpt-oss-120b",
    temperature=0.1,
    max_tokens=1024
)

In [93]:
# generate out retrival-augmented output
def generate_output(query,rag_retriever,llm,top_k=3):
    results=rag_retriever.retrieve(query,top_k)

    context="\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant content for your input")

    #context+query
    prompt= f""" use given context to generate the answer for the query
                Context:{context}
                Query:{query}"""

    response=llm.invoke([prompt.format(context=context,query=query)]) #expecting a list as prompt
    
    return response.text

In [100]:
answer=generate_output("What is ENCODER",rag_retriever,llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 0 documents
we found no relevant content for your input


In [101]:
print(answer)

**Encoder – a brief overview**

An **encoder** is a component (hardware or software) that transforms data from one representation into another, usually to make it more suitable for storage, transmission, or further processing. The exact meaning varies by field, but the core idea is the same: *convert input information into a coded form*.

| Domain | What the encoder does | Typical purpose |
|--------|----------------------|-----------------|
| **Digital electronics / sensors** | Converts a physical quantity (e.g., rotation, position, light intensity) into a digital signal (binary code). | Provides the computer or controller with measurable feedback (e.g., rotary encoders on motors). |
| **Data compression** | Maps raw data (text, audio, video, images) to a more compact representation using algorithms like Huffman coding, JPEG, MP3, H.264, etc. | Reduces file size for storage or faster transmission. |
| **Communication systems** | Transforms a bitstream into a waveform (e.g., line codin